### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [39]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [48]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")


Found 1 PDF files to process

Processing: AI-for-Education-RAG.pdf
  ✓ Loaded 18 pages

Total documents loaded: 18


In [57]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [58]:
chunks = split_documents(all_pdf_documents)
chunks

Split 18 documents into 18 chunks

Example chunk:
Content: Using your own content in LLM’s -
Retrieval Augmented Generation
(RAG)...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-03-04T11:16:13+00:00', 'moddate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'AI-for-Education-RAG.pdf', 'file_type': 'pdf'}


Split 18 documents into 18 chunks

Example chunk:
Content: Using your own content in LLM’s -
Retrieval Augmented Generation
(RAG)...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-03-04T11:16:13+00:00', 'moddate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'AI-for-Education-RAG.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-03-04T11:16:13+00:00', 'moddate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'AI-for-Education-RAG.pdf', 'file_type': 'pdf'}, page_content='Using your own content in LLM’s -\nRetrieval Augmented Generation\n(RAG)'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2024-03-04T11:16:13+00:00', 'moddate': '2024-03-04T11:16:13+00:00', 'source': '../data/pdf/AI-for-Education-RAG.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2', 'source_file': 'AI-for-Education-RAG.pdf', 'file_type': 'pdf'}, page_content='Gen-AI can help with education in many ways\nSaves time\n• AI lesson planning reduces the time teachers spend preparing lessons. \n• Can automate marking using AI – image recognition or voice technologies to assess students as \nthey read aloud.\nImproved quality\n• Adapt high 

### Embedding and Vector Store DB


In [53]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [54]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7049.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7049.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store


In [55]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            import os
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [59]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 18 texts...


Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.93s/it]

Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.93s/it]

Generated embeddings with shape: (18, 384)
Adding 18 documents to vector store...
Successfully added 18 documents to vector store
Total documents in collection: 18


### Retrival Pipeline

In [60]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [64]:
rag_retriever.retrieve("The training data for major LLM’s isn’t specific to any one country and often doesn’t include your education resources")


Retrieving documents for query: 'The training data for major LLM’s isn’t specific to any one country and often doesn’t include your education resources'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Retrieving documents for query: 'The training data for major LLM’s isn’t specific to any one country and often doesn’t include your education resources'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]

Retrieving documents for query: 'The training data for major LLM’s isn’t specific to any one country and often doesn’t include your education resources'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Retrieving documents for query: 'The training data for major LLM’s isn’t specific to any one country and often doesn’t include your education resources'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_24c5e0e6_2',
  'content': 'But many of these require introducing your own \ninformation into the models. \nThe training data for major LLM’s isn’t specific to any one country and often \ndoesn’t include your education resources\nIf your content isn’t in the training data it can only ever be approximated in \nthe outputs. \nSo how you integrate local information – like your own teacher guides?',
  'metadata': {'file_type': 'pdf',
   'doc_index': 2,
   'page': 2,
   'moddate': '2024-03-04T11:16:13+00:00',
   'producer': 'PyPDF',
   'creator': 'PyPDF',
   'total_pages': 18,
   'content_length': 362,
   'source_file': 'AI-for-Education-RAG.pdf',
   'creationdate': '2024-03-04T11:16:13+00:00',
   'page_label': '3',
   'source': '../data/pdf/AI-for-Education-RAG.pdf'},
  'similarity_score': 0.4918460249900818,
  'distance': 0.5081539750099182,
  'rank': 1}]

### Integrate VectorDB Context Pipeline with LLM Output

In [70]:
# OPTION 1: Use Groq API (Free & Fast) - RECOMMENDED
# Install: pip install groq

import os
from groq import Groq

# Get your API key from https://console.groq.com
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # Set as environment variable

if not GROQ_API_KEY:
    print("⚠ GROQ_API_KEY not found!")
    print("Set it in terminal: export GROQ_API_KEY='your-key-here'")
    print("Or pass it directly below:")
    # GROQ_API_KEY = "your-key-here"
else:
    print("✓ Groq API key found")

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Available models: mixtral-8x7b-32768, llama2-70b-4096, gemma-7b-it
groq_model = "mixtral-8x7b-32768"  # Fast and capable

print(f"✓ Groq client initialized with model: {groq_model}")


ModuleNotFoundError: No module named 'groq'

In [ ]:
# Create a Groq LLM wrapper compatible with our RAG pipeline
class GroqLLM:
    """Wrapper for Groq API to match OllamaLLM interface"""
    
    def __init__(self, api_key: str, model: str = "mixtral-8x7b-32768", temperature: float = 0.7):
        """
        Initialize Groq LLM
        
        Args:
            api_key: Groq API key
            model: Model to use
            temperature: Temperature for generation
        """
        self.client = Groq(api_key=api_key)
        self.model = model
        self.temperature = temperature
    
    def invoke(self, prompt: str) -> str:
        """Generate response from prompt"""
        try:
            message = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=self.temperature,
                max_tokens=2048
            )
            return message.choices[0].message.content
        except Exception as e:
            return f"Error: {e}"

# Initialize Groq LLM if API key is available
if GROQ_API_KEY:
    llm = GroqLLM(api_key=GROQ_API_KEY, model="mixtral-8x7b-32768")
    print("✓ Groq LLM ready to use")
else:
    llm = None
    print("⚠ Groq API key not set. Cannot initialize LLM.")


### Quick Setup Guide for Free LLM APIs

**Option A: Groq (Recommended)**
```bash
# 1. Install Groq client
pip install groq

# 2. Get free API key from https://console.groq.com

# 3. Set environment variable (macOS/Linux)
export GROQ_API_KEY="your-api-key-here"

# 4. Run the Groq initialization cells above
```

**Option B: Together AI**
```bash
pip install together
export TOGETHER_API_KEY="your-key-here"
```

**Option C: Hugging Face**
```bash
pip install huggingface-hub
export HF_TOKEN="your-token-here"
```

**Groq Free Tier:**
- 10,000 tokens/day
- No credit card required initially
- Ultra-fast inference (great for RAG)
- Available models: Mixtral-8x7b, Llama2-70b, Gemma-7b


In [ ]:
# Test Groq connection
if llm:
    print("Testing Groq API connection...")
    try:
        test_response = llm.invoke("What is machine learning in one sentence?")
        print("✓ Groq connection successful!")
        print(f"\nResponse:\n{test_response}")
    except Exception as e:
        print(f"✗ Error: {e}")
else:
    print("⚠ LLM not initialized. Please set GROQ_API_KEY first.")


### Ollama Integration for RAG Pipeline

In [ ]:
# Import Ollama and RAG components
from langchain_ollama import OllamaLLM

# Initialize Ollama with a local model
# Make sure ollama serve is running on localhost:11434
llm = OllamaLLM(
    model="mistral",  # or "llama2", "neural-chat", etc.
    base_url="http://localhost:11434",
    temperature=0.7,
    num_ctx=2048
)

print("✓ Ollama LLM initialized successfully")
print(f"Model: mistral")
print(f"Base URL: http://localhost:11434")


In [ ]:
# Test Ollama connection
try:
    test_response = llm.invoke("What is machine learning in one sentence?")
    print("✓ Ollama connection successful!")
    print(f"\nTest response:\n{test_response}")
except Exception as e:
    print(f"✗ Error connecting to Ollama: {e}")
    print("Make sure Ollama is running: 'ollama serve'")


In [ ]:

# First, check which models are available and pull if needed
import subprocess

def get_available_models():
    """Get list of models available in Ollama"""
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            timeout=10
        )
        return result.stdout
    except Exception as e:
        return f"Error listing models: {e}"

def pull_model(model_name: str):
    """Pull a model from Ollama"""
    try:
        print(f"Pulling {model_name}... This may take a few minutes on first pull.")
        result = subprocess.run(
            ["ollama", "pull", model_name],
            capture_output=True,
            text=True,
            timeout=600  # 10 minute timeout
        )
        print(result.stdout)
        if result.returncode == 0:
            print(f"✓ Successfully pulled {model_name}")
            return True
        else:
            print(f"✗ Error pulling model: {result.stderr}")
            return False
    except subprocess.TimeoutExpired:
        print(f"✗ Timeout: Model pull took too long")
        return False
    except Exception as e:
        print(f"✗ Error: {e}")
        return False

# Check available models
print("Available Ollama models:")
print(get_available_models())

# If mistral is not available, pull it or use neural-chat
print("\n" + "="*80)
print("PULLING REQUIRED MODEL...")
print("="*80)

model_to_use = "mistral"  # Change to "neural-chat" or "llama2" if preferred
pull_model(model_to_use)
# Create a custom prompt for RAG with context
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following pieces of context to answer the question. 
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""
)

print("✓ RAG prompt template created")


In [ ]:
# Create a complete RAG pipeline using existing retriever + Ollama LLM
class OllamaRAGPipeline:
    """Complete RAG pipeline combining retrieval and generation"""
    
    def __init__(self, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 3):
        """
        Initialize RAG pipeline
        
        Args:
            retriever: RAGRetriever instance for document retrieval
            llm: OllamaLLM instance for text generation
            top_k: Number of context documents to retrieve
        """
        self.retriever = retriever
        self.llm = llm
        self.top_k = top_k
    
    def format_context(self, retrieved_docs: List[Dict]) -> str:
        """Format retrieved documents into context string"""
        if not retrieved_docs:
            return "No relevant documents found."
        
        context_parts = []
        for i, doc in enumerate(retrieved_docs, 1):
            context_parts.append(
                f"Document {i} (Similarity: {doc['similarity_score']:.2%}):\n"
                f"{doc['content']}\n"
                f"Source: {doc['metadata'].get('source_file', 'Unknown')}\n"
            )
        
        return "\n---\n".join(context_parts)
    
    def generate_response(self, query: str, use_context: bool = True) -> Dict[str, Any]:
        """
        Generate response using retrieval + generation
        
        Args:
            query: User query
            use_context: Whether to use retrieved context
            
        Returns:
            Dictionary with response, context, and metadata
        """
        print(f"\n{'='*80}")
        print(f"Query: {query}")
        print(f"{'='*80}\n")
        
        # Step 1: Retrieve relevant documents
        print("Step 1: Retrieving relevant documents...")
        retrieved_docs = self.retriever.retrieve(query, top_k=self.top_k)
        
        if not retrieved_docs:
            print("⚠ No documents retrieved!")
        else:
            print(f"✓ Retrieved {len(retrieved_docs)} documents")
        
        # Step 2: Format context
        print("\nStep 2: Formatting context...")
        context = self.format_context(retrieved_docs)
        print(f"Context length: {len(context)} characters")
        
        # Step 3: Build prompt with context
        print("\nStep 3: Building prompt for LLM...")
        if use_context and retrieved_docs:
            prompt = f"""You are a helpful AI assistant that answers questions based on provided documents.

CONTEXT (from relevant documents):
{context}

QUESTION: {query}

INSTRUCTIONS:
- Answer based ONLY on the context provided above
- If the answer is not in the context, say "The provided documents don't contain information about this"
- Be specific and cite which document the answer comes from
- Keep your answer concise and relevant

ANSWER:"""
        else:
            prompt = f"""Answer the following question:

QUESTION: {query}

ANSWER:"""
        
        # Step 4: Generate response with Ollama
        print("\nStep 4: Generating response with Ollama (Mistral)...")
        print("-" * 80)
        response = self.llm.invoke(prompt)
        print("-" * 80)
        
        return {
            "query": query,
            "response": response,
            "context": context,
            "retrieved_docs": retrieved_docs,
            "num_docs_retrieved": len(retrieved_docs)
        }

# Initialize the RAG pipeline
rag_pipeline = OllamaRAGPipeline(
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)

print("✓ Complete RAG pipeline initialized with:")
print("  - Existing RAGRetriever (semantic search)")
print("  - Ollama LLM (Mistral model)")
print("  - Custom context formatting")
